In [ ]:
# Feature Engineering for Multi-Disease Prediction

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

print("=== Phase 4: Feature Engineering ===\n")

# Load the district climate data
DATA_PROCESSED = r"I:\Data Science Projects\disease_prediction\data\processed"
df = pd.read_csv(os.path.join(DATA_PROCESSED, "district_monthly_climate_2010_2024.csv"))

print(f"Loaded {len(df):,} records")
print(f"Districts: {df['district'].nunique()}")
print(f"Time range: {df['date'].min()} to {df['date'].max()}\n")

# Convert date
df['date'] = pd.to_datetime(df['date'])

# FEATURE ENGINEERING 
# 1. Lag Features (previous months)
for lag in [1, 2, 3]:
    df[f't2m_lag{lag}'] = df.groupby('district')['t2m'].shift(lag)
    df[f'tp_lag{lag}'] = df.groupby('district')['tp'].shift(lag)

# 2. Rolling Averages (3-month and 6-month)
df['t2m_roll3'] = df.groupby('district')['t2m'].rolling(3).mean().reset_index(0, drop=True)
df['tp_roll3']  = df.groupby('district')['tp'].rolling(3).mean().reset_index(0, drop=True)

# 3. Anomalies (deviation from district's long-term monthly average)
df['month'] = df['date'].dt.month
df['t2m_clim'] = df.groupby(['district', 'month'])['t2m'].transform('mean')
df['tp_clim']  = df.groupby(['district', 'month'])['tp'].transform('mean')

df['t2m_anomaly'] = df['t2m'] - df['t2m_clim']
df['tp_anomaly']  = df['tp'] - df['tp_clim']

# 4. Interaction term
df['temp_precip_interaction'] = df['t2m'] * df['tp']

# 5. Seasonal indicators
df['quarter'] = df['date'].dt.quarter
df['is_wet_season'] = df['month'].isin([3,4,5,9,10,11]).astype(int)  # Rough wet season for Uganda

print("Feature engineering completed!")
print(f"Total features now: {len(df.columns)}")
print("\nNew features added:")
print([col for col in df.columns if col not in ['date', 'district', 't2m', 'tp']])

# Save final engineered dataset
final_file = os.path.join(DATA_PROCESSED, "district_monthly_climate_features_2010_2024.csv")
df.to_csv(final_file, index=False)

print(f"\n Final engineered dataset saved")